# A/B Testing Reminder Emails to Increase Admissions Quiz Completion: A MongoDB Pipeline Using Python (Pandas, PyMongo, and a MongoRepository Class)

## Latihan konsep dasar

1. pd.to_datetime mengubah string “2022-05-02” menjadi Timestamp. pd.DateOffset(days=1) menambah satu hari sehingga diperoleh rentang waktu 2 sampai 3 Mei 2022.

In [43]:
import pandas as pd

# Convert a date string to a datetime object
start = pd.to_datetime("2022-05-02", format="%Y-%m-%d")
# Offset by one day
end = start + pd.DateOffset(days=1)

print("start:", start)
print("end  :", end)
print("type :", type(start))

start: 2022-05-02 00:00:00
end  : 2022-05-03 00:00:00
type : <class 'pandas.Timestamp'>


2. Contoh query dict MongoDB: createdAt ≥ start dan < end, serta admissionsQuiz == "incomplete". Belum dijalankan ke database.

In [44]:
# Example query dict (not executed against a database yet)
example_query = {
    "createdAt": {"$gte": start, "$lt": end},
    "admissionsQuiz": "incomplete",
}
print(example_query)

{'createdAt': {'$gte': Timestamp('2022-05-02 00:00:00'), '$lt': Timestamp('2022-05-03 00:00:00')}, 'admissionsQuiz': 'incomplete'}


3. Mengacak 10 ID dengan random.seed(42) lalu membaginya dua menjadi control dan treatment (masing-masing 5).

In [45]:
import random

demo_ids = list(range(10))
random.seed(42)
random.shuffle(demo_ids)
print("Shuffled:", demo_ids)

mid = len(demo_ids) // 2
control = demo_ids[:mid]
treatment = demo_ids[mid:]
print("Control  :", control)
print("Treatment:", treatment)

Shuffled: [7, 3, 2, 8, 5, 6, 9, 4, 0, 1]
Control  : [7, 3, 2, 8, 5]
Treatment: [6, 9, 4, 0, 1]


4. Komentar demonstrasi update_one dengan $set serta arti matched_count dan modified_count.

In [46]:
# Demonstration (not connected to a real database)
# result = collection.update_one(
#     {"_id": "abc123"},
#     {"$set": {"inExperiment": True, "group": "no email (control)"}},
# )
# result.matched_count   -> 1 if the filter matched a document
# result.modified_count  -> 1 if the document was actually changed

5. Kelas Greeter dengan __init__ dan metode greet() sebagai latihan dasar OOP.

In [47]:
class Greeter:
    """A minimal class to demonstrate __init__ and methods."""

    def __init__(self, name="World"):
        # `self.name` is an *instance attribute*
        self.name = name

    def greet(self):
        """Return a greeting string."""
        return f"Hello, {self.name}!"


g = Greeter("Data Science")
print(g.greet())
print("Attribute:", g.name)

Hello, Data Science!
Attribute: Data Science


6. dir() menampilkan atribut publik objek (greet, name).

In [48]:
# dir() on a simple object to illustrate
print([attr for attr in dir(g) if not attr.startswith("_")])

['greet', 'name']


## Proyek utama dengan MongoDB

1. Mengimpor random, pandas, MongoClient, dan reset, serta mendefinisikan alamat host MongoDB.

In [49]:
import random

import pandas as pd

from pymongo import MongoClient
from wqulibs.database import reset

MONGODB_HOST = "localhost"

2. Membuat MongoClient, memilih database wqu-abtest dan koleksi ds-applicants, lalu me-reset koleksi.

In [50]:
client = MongoClient(host=MONGODB_HOST, port=27017)
ds_app = client["wqu-abtest"]["ds-applicants"]
reset(ds_app)
print("client:", type(client))
print("ds_app:", type(ds_app))

client: <class 'pymongo.synchronous.mongo_client.MongoClient'>
ds_app: <class 'pymongo.synchronous.collection.Collection'>


3. assert memastikan client dan ds_app bertipe benar. Koneksi berhasil.

In [51]:
assert "MongoClient" in str(type(client)), (
    f"Expected MongoClient, got {type(client)}"
)
assert str(type(ds_app)).endswith("Collection'>"), (
    f"Expected a Collection, got {type(ds_app)}"
)
print("\u2705 Connection established successfully.")

✅ Connection established successfully.


4. aggregate dengan $group menghitung jumlah per status kuis: selesai 3.717 dan belum selesai 1.308.

In [52]:
# group by admissionsQuiz and count each status
result = ds_app.aggregate(
    [{"$group": {"_id": "$admissionsQuiz", "count": {"$sum": 1}}}]
)
for doc in result:
    if doc["_id"] == "incomplete":
        incomplete = doc["count"]
    else:
        complete = doc["count"]

print("Completed quiz:", complete)
print("Did not complete quiz:", incomplete)

Completed quiz: 3717
Did not complete quiz: 1308


5. Menghitung proporsi yang belum menyelesaikan kuis, yaitu 0,26 (26%).

In [53]:
total = complete + incomplete
prop_incomplete = incomplete / total
print(
    "Proportion who did not complete quiz:",
    round(prop_incomplete, 2),
)

Proportion who did not complete quiz: 0.26


6. Menulis hipotesis nol (H0: email tidak meningkatkan penyelesaian) dan hipotesis alternatif (Ha: email meningkatkan penyelesaian).

In [54]:
null_hypothesis = (
    "There is no relationship between receiving a reminder"
    " email and completing the admissions quiz. Sending the"
    " email does not increase the completion rate."
)
alternate_hypothesis = (
    "There is a relationship between receiving a reminder"
    " email and completing the admissions quiz. Sending the"
    " email does increase the completion rate."
)
print("H0:", null_hypothesis)
print("Ha:", alternate_hypothesis)

H0: There is no relationship between receiving a reminder email and completing the admissions quiz. Sending the email does not increase the completion rate.
Ha: There is a relationship between receiving a reminder email and completing the admissions quiz. Sending the email does increase the completion rate.


7. assert validasi jumlah, proporsi, dan hipotesis. Tingkat penyelesaian 74% dan tingkat belum selesai 26%.

In [55]:
assert isinstance(complete, int) and complete > 0, (
    f"Expected `complete` to be a positive int, got {complete!r}"
)
assert isinstance(incomplete, int) and incomplete > 0, (
    f"Expected `incomplete` to be a positive int, got {incomplete!r}"
)
assert 0 < prop_incomplete < 1, (
    f"Expected proportion between 0 and 1, got {prop_incomplete}"
)
assert isinstance(null_hypothesis, str) and len(null_hypothesis) > 10, (
    "Write a descriptive null hypothesis string."
)
assert isinstance(alternate_hypothesis, str) and len(alternate_hypothesis) > 10, (
    "Write a descriptive alternate hypothesis string."
)
print(f"\u2705 Completion rate: {1 - prop_incomplete:.1%} complete, "
      f"{prop_incomplete:.1%} incomplete.")

✅ Completion rate: 74.0% complete, 26.0% incomplete.


8. Fungsi find_by_date() mencari pendaftar yang belum menyelesaikan kuis pada tanggal tertentu.

In [56]:
def find_by_date(collection, date_string):
    """Find no-quiz applicants created on a given date.

    Parameters
    ----------
    collection : pymongo.collection.Collection
        Collection to search.
    date_string : str
        Date in '%Y-%m-%d' format, e.g. '2022-06-28'.

    Returns
    -------
    observations : list
        List of matching documents (dictionaries).
    """
    start = pd.to_datetime(date_string, format="%Y-%m-%d")
    end = start + pd.DateOffset(days=1)
    query = {
        "createdAt": {"$gte": start, "$lt": end},
        "admissionsQuiz": "incomplete",
    }
    result = collection.find(query)
    observations = list(result)
    return observations

9. Memanggil fungsi untuk 2022-05-02. Hasilnya 49 dokumen pendaftar, dan contoh dokumen pertama ditampilkan.

In [57]:
observations = find_by_date(
    ds_app, # pass the collection
    date_string="2022-05-02") # use "2022-05-02"
print("observations type:", type(observations))
print("observations len:", len(observations))
observations[0]

observations type: <class 'list'>
observations len: 49


{'_id': ObjectId('6ae3fb4e4fe4093c448b70a8'),
 'createdAt': datetime.datetime(2022, 5, 2, 2, 0, 11),
 'firstName': 'Virginia',
 'lastName': 'Anderson',
 'email': 'virginia.anderson18@yahow.com',
 'birthday': datetime.datetime(1998, 5, 17, 0, 0),
 'gender': 'female',
 'highestDegreeEarned': "Bachelor's degree",
 'countryISO2': 'SL',
 'admissionsQuiz': 'incomplete'}

10. assert validasi daftar hasil (tidak kosong, berupa dict, punya _id).

In [58]:
assert isinstance(observations, list), (
    f"Expected a list, got {type(observations)}"
)
assert len(observations) > 0, (
    "Expected at least one observation for 2022-05-02."
)
assert isinstance(observations[0], dict), (
    f"Expected list of dicts, got list of {type(observations[0])}"
)
assert "_id" in observations[0], (
    "Each document should contain an '_id' key."
)
print(f"\u2705 Extracted {len(observations)} no-quiz applicants "
      f"for 2022-05-02.")

✅ Extracted 49 no-quiz applicants for 2022-05-02.


11. assign_to_groups() mengacak (seed=42) lalu membagi dua: setengah pertama ke grup kontrol (tanpa email) dan sisanya ke treatment (dengan email). Kolom inExperiment dan group ditambahkan.

In [59]:
def assign_to_groups(observations):
    """Randomly assign observations to control and treatment groups.

    Parameters
    ----------
    observations : list
        List of applicant documents.

    Returns
    -------
    observations : list
        Same list with 'inExperiment' and 'group' keys added.
    """
    random.seed(42)
    random.shuffle(observations)
    # find the midpoint index
    idx = len(observations) // 2
    for doc in observations[:idx]:
        doc["inExperiment"] = True
        doc["group"] = "no email (control)"
    for doc in observations[idx:]:
        doc["inExperiment"] = True
        doc["group"] = "email (treatment)"
    return observations

12. Menjalankan fungsi tersebut. Hasilnya 49 dokumen dengan atribut baru.

In [60]:
observations_assigned = assign_to_groups(observations)
print("observations_assigned type:", type(observations_assigned))
print("observations_assigned len:", len(observations_assigned))
observations_assigned[0]

observations_assigned type: <class 'list'>
observations_assigned len: 49


{'_id': ObjectId('6ae3fb4e4fe4093c448b7934'),
 'createdAt': datetime.datetime(2022, 5, 2, 18, 57, 15),
 'firstName': 'Lamar',
 'lastName': 'Powell',
 'email': 'lamar.powell93@hotmeal.com',
 'birthday': datetime.datetime(1999, 8, 11, 0, 0),
 'gender': 'male',
 'highestDegreeEarned': "Master's degree",
 'countryISO2': 'US',
 'admissionsQuiz': 'incomplete',
 'inExperiment': True,
 'group': 'no email (control)'}

13. assert validasi: ada 2 grup dan selisih ukuran ≤ 1 (24 kontrol dan 25 treatment).

In [61]:
assert isinstance(observations_assigned, list), (
    f"Expected a list, got {type(observations_assigned)}"
)
assert "inExperiment" in observations_assigned[0], (
    "Documents should have an 'inExperiment' key."
)
assert "group" in observations_assigned[0], (
    "Documents should have a 'group' key."
)
groups = {doc["group"] for doc in observations_assigned}
assert groups == {"no email (control)", "email (treatment)"}, (
    f"Expected two groups, got {groups}"
)
n_control = sum(
    1 for d in observations_assigned
    if d["group"] == "no email (control)"
)
n_treat = sum(
    1 for d in observations_assigned
    if d["group"] == "email (treatment)"
)
assert abs(n_control - n_treat) <= 1, (
    f"Groups should be roughly equal. Control: {n_control}, "
    f"Treatment: {n_treat}"
)
print(f"\u2705 Assigned {n_control} to control and "
      f"{n_treat} to treatment.")

✅ Assigned 24 to control and 25 to treatment.


14. export_treatment_emails() mengekspor email grup treatment (kolom email dan tag="ab-test") ke CSV bertanggal.

In [62]:
def export_treatment_emails(
    observations_assigned, directory="."
):
    """Export treatment-group emails to a dated CSV file.

    Parameters
    ----------
    observations_assigned : list
        Documents with group assignment.
    directory : str, default='.'
        Output directory for the CSV file.

    Returns
    -------
    None
    """
    df = pd.DataFrame(observations_assigned)
    df["tag"] = "ab-test"  # add a "tag" column
    # filter for treatment group only
    mask = df["group"] == "email (treatment)"
    datestring = pd.Timestamp.now().strftime("%Y-%m-%d")
    filename = directory + "/" + datestring + "_ab-test.csv"
    # save only "email" and "tag" columns
    df[mask][["email", "tag"]].to_csv(filename, index=False)

export_treatment_emails(observations_assigned)

15. assert validasi CSV: ada kolom email dan tag, dengan 25 email treatment.

In [63]:
import os

datestring = pd.Timestamp.now().strftime("%Y-%m-%d")
expected_file = f"./{datestring}_ab-test.csv"
assert os.path.exists(expected_file), (
    f"Expected CSV file '{expected_file}' not found."
)
email_df = pd.read_csv(expected_file)
assert "email" in email_df.columns, (
    f"CSV should have an 'email' column, got {list(email_df.columns)}"
)
assert "tag" in email_df.columns, (
    f"CSV should have a 'tag' column, got {list(email_df.columns)}"
)
assert (email_df["tag"] == "ab-test").all(), (
    "Every row in the 'tag' column should be 'ab-test'."
)
print(f"\u2705 CSV exported with {len(email_df)} treatment emails.")

✅ CSV exported with 25 treatment emails.


16. Mengambil satu pendaftar (applicant_id) lalu find_one untuk melihat dokumen aslinya di database (belum ada group).

In [64]:
updated_applicant = observations_assigned[0]
applicant_id = updated_applicant["_id"]
print("applicant_id type:", type(applicant_id))
print("applicant_id:", applicant_id)
ds_app.find_one({"_id": applicant_id})

applicant_id type: <class 'bson.objectid.ObjectId'>
applicant_id: 6ae3fb4e4fe4093c448b7934


{'_id': ObjectId('6ae3fb4e4fe4093c448b7934'),
 'createdAt': datetime.datetime(2022, 5, 2, 18, 57, 15),
 'firstName': 'Lamar',
 'lastName': 'Powell',
 'email': 'lamar.powell93@hotmeal.com',
 'birthday': datetime.datetime(1999, 8, 11, 0, 0),
 'gender': 'male',
 'highestDegreeEarned': "Master's degree",
 'countryISO2': 'US',
 'admissionsQuiz': 'incomplete'}

17. update_one dengan $set menyimpan data grup ke database (matched 1, modified 1).

In [65]:
result = ds_app.update_one(
    {"_id": applicant_id},  # filter by _id
    {"$set": updated_applicant}, # fields to update
)
print("matched:", result.matched_count)
print("modified:", result.modified_count)

matched: 1
modified: 1


18. Menampilkan atribut hasil update dan raw_result.

In [66]:
# list public attributes (filter out names starting with "_")
print([a for a in dir(result) if not a.startswith("_")])
# Inspect raw_result
print(result.raw_result)

['acknowledged', 'did_upsert', 'matched_count', 'modified_count', 'raw_result', 'upserted_id']
{'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}


19. Fungsi update_applicants() 

In [67]:
def update_applicants(collection, observations_assigned):
    """Update applicant documents in a MongoDB collection.

    Parameters
    ----------
    collection : pymongo.collection.Collection
        Target collection.
    observations_assigned : list
        Documents with group assignments to write back.

    Returns
    -------
    transaction_result : dict
        Keys 'n' (matched) and 'nModified' (changed).
    """
    n = 0
    n_modified = 0
    for doc in observations_assigned:
        result = collection.update_one(
            {"_id": doc["_id"]}, # filter by document _id
            {"$set": doc}, # update with the full doc
        )
        n += result.matched_count
        n_modified += result.modified_count
    transaction_result = {"n": n, "nModified": n_modified}
    return transaction_result

20. Fungsi update_applicants() meng-update semua dokumen dengan loop. Hasilnya n = 49 dan nModified = 48 (satu sudah ter-update sebelumnya).

In [68]:
result = update_applicants(ds_app, observations_assigned)
print("result type:", type(result))
result

result type: <class 'dict'>


{'n': 49, 'nModified': 48}

21. assert validasi hasil update.

In [69]:
assert isinstance(result, dict), (
    f"Expected a dict, got {type(result)}"
)
assert "n" in result and "nModified" in result, (
    f"Expected keys 'n' and 'nModified', got {list(result.keys())}"
)
assert result["n"] == len(observations_assigned), (
    f"Expected n={len(observations_assigned)}, got n={result['n']}"
)
print(f"\u2705 Updated {result['n']} documents "
      f"({result['nModified']} modified).")

✅ Updated 49 documents (48 modified).


22. Kelas MongoRepository membungkus akses data: find_by_date, update_applicants, dan assign_to_groups (ekstrak, transformasi, dan muat dalam satu metode).

In [70]:
class MongoRepository:
    """Repository class for interacting with MongoDB.

    Parameters
    ----------
    client : pymongo.MongoClient
        Default: MongoClient(host=MONGODB_HOST, port=27017).
    db : str
        Default: 'wqu-abtest'.
    collection : str
        Default: 'ds-applicants'.

    Attributes
    ----------
    collection : pymongo.collection.Collection
        Collection for extraction and loading.
    """

    def __init__(
        self,
        client=MongoClient(host=MONGODB_HOST, port=27017),
        db="wqu-abtest",
        collection="ds-applicants",
    ):
        # store the resolved collection as an attribute
        self.collection = client[db][collection]

    def find_by_date(self, date_string):
        # reuse the same logic from the standalone function
        # but use self.collection instead of a parameter
        start = pd.to_datetime(date_string, format="%Y-%m-%d")
        end = start + pd.DateOffset(days=1)
        query = {
            "createdAt": {"$gte": start, "$lt": end},
            "admissionsQuiz": "incomplete",
        }
        result = self.collection.find(query)
        return list(result)

    def update_applicants(self, documents):
        n = 0
        n_modified = 0
        for doc in documents:
            result = self.collection.update_one(
                {"_id": doc["_id"]},
                {"$set": doc},
            )
            n += result.matched_count
            n_modified += result.modified_count
        return {"n": n, "nModified": n_modified}

    def assign_to_groups(self, date_string):
        # extract
        observations = self.find_by_date(date_string)
        # transform: shuffle and split
        random.seed(42)
        random.shuffle(observations)
        idx = len(observations) // 2
        for doc in observations[:idx]:
            doc["inExperiment"] = True
            doc["group"] = "no email (control)"
        for doc in observations[idx:]:
            doc["inExperiment"] = True
            doc["group"] = "email (treatment)"
        # load
        result = self.update_applicants(observations)
        return result

23. Membuat instance repo dan mengecek tipe koleksinya.

In [71]:
repo = MongoRepository()
print("repo type:", type(repo))
c_test = repo.collection
print("collection type:", type(c_test))

repo type: <class '__main__.MongoRepository'>
collection type: <class 'pymongo.synchronous.collection.Collection'>


24. repo.find_by_date("2022-05-15") menghasilkan 30 pendaftar yang belum menyelesaikan kuis.

In [72]:
may_15_users = repo.find_by_date("2022-05-15")
print("may_15_users type:", type(may_15_users))
print("may_15_users len:", len(may_15_users))
may_15_users[:3]

may_15_users type: <class 'list'>
may_15_users len: 30


[{'_id': ObjectId('6ae3fb4e4fe4093c448b6fbd'),
  'createdAt': datetime.datetime(2022, 5, 15, 20, 21, 12),
  'firstName': 'Patrick',
  'lastName': 'Derosa',
  'email': 'patrick.derosa81@hotmeal.com',
  'birthday': datetime.datetime(2000, 9, 30, 0, 0),
  'gender': 'male',
  'highestDegreeEarned': "Bachelor's degree",
  'countryISO2': 'UA',
  'admissionsQuiz': 'incomplete'},
 {'_id': ObjectId('6ae3fb4e4fe4093c448b70dc'),
  'createdAt': datetime.datetime(2022, 5, 15, 10, 50, 56),
  'firstName': 'Deidre',
  'lastName': 'Pagan',
  'email': 'deidre.pagan75@hotmeal.com',
  'birthday': datetime.datetime(1996, 12, 2, 0, 0),
  'gender': 'female',
  'highestDegreeEarned': "Bachelor's degree",
  'countryISO2': 'ZW',
  'admissionsQuiz': 'incomplete'},
 {'_id': ObjectId('6ae3fb4e4fe4093c448b716f'),
  'createdAt': datetime.datetime(2022, 5, 15, 5, 8, 35),
  'firstName': 'Harry',
  'lastName': 'Ellis',
  'email': 'harry.ellis78@microsift.com',
  'birthday': datetime.datetime(2000, 2, 6, 0, 0),
  'gende

25. repo.update_applicants(...) dijalankan ulang, hasilnya n = 49 dan nModified = 0 (data sudah sama).

In [73]:
result = repo.update_applicants(observations_assigned)
print("result type:", type(result))
result

result type: <class 'dict'>


{'n': 49, 'nModified': 0}

26. repo.assign_to_groups("2022-05-14") menghasilkan n = 44 dan nModified = 44.

In [74]:
result = repo.assign_to_groups("2022-05-14")
print("result type:", type(result))
result

result type: <class 'dict'>


{'n': 44, 'nModified': 44}

27. assert akhir dan uji end-to-end untuk 2022-05-16 (38 pendaftar), termasuk memastikan kedua grup ada.

In [75]:
assert isinstance(repo, MongoRepository), (
    f"Expected MongoRepository instance, got {type(repo)}"
)
assert hasattr(repo, "collection"), (
    "MongoRepository should have a 'collection' attribute."
)
assert hasattr(repo, "find_by_date"), (
    "MongoRepository should have a 'find_by_date' method."
)
assert hasattr(repo, "update_applicants"), (
    "MongoRepository should have an 'update_applicants' method."
)
assert hasattr(repo, "assign_to_groups"), (
    "MongoRepository should have an 'assign_to_groups' method."
)
# End-to-end test with a fresh date
repo_test = MongoRepository()
test_result = repo_test.assign_to_groups("2022-05-16")
assert isinstance(test_result, dict), (
    f"assign_to_groups should return a dict, got {type(test_result)}"
)
assert test_result["n"] > 0, (
    "Expected at least one matched document for 2022-05-16."
)
may_16_docs = repo_test.find_by_date("2022-05-16")
groups_found = {d.get("group") for d in may_16_docs}
assert "no email (control)" in groups_found, (
    "Expected control group in 2022-05-16 documents."
)
assert "email (treatment)" in groups_found, (
    "Expected treatment group in 2022-05-16 documents."
)
print(f"\u2705 MongoRepository class works end-to-end. "
      f"Assigned {test_result['n']} applicants for 2022-05-16.")

✅ MongoRepository class works end-to-end. Assigned 38 applicants for 2022-05-16.


Kesimpulan Singkat

- Tujuan: Menyiapkan infrastruktur eksperimen A/B untuk menguji apakah email pengingat meningkatkan penyelesaian kuis admisi pada pendaftar program data science.
- Kondisi awal: Dari 5.025 pendaftar, 3.717 (74%) menyelesaikan kuis dan 1.308 (26%) belum.
- Desain eksperimen: Pendaftar yang belum menyelesaikan kuis pada satu hari tertentu diacak (seed=42) menjadi dua grup. Grup kontrol tidak menerima email dan grup treatment menerima email pengingat. H0 menyatakan email tidak berpengaruh, H1 menyatakan email meningkatkan penyelesaian.
- Pipeline ETL: Kode disusun dalam tiga tahap: extract (find_by_date), transform (acak dan beri label grup), load (update_applicants ke MongoDB, plus ekspor CSV email grup treatment).
- Refaktor ke OOP: Fungsi-fungsi dikemas dalam kelas MongoRepository agar reusable dan mudah dijalankan harian. Uji pada tanggal 14 dan 16 Mei 2022 berhasil (44 dan 38 pendaftar).
- Belum ada hasil uji hipotesis: Notebook ini baru mencakup persiapan dan pengacakan. Tingkat penyelesaian kuis grup kontrol vs treatment belum dibandingkan, jadi belum ada kesimpulan apakah email berpengaruh.

Catatan kritis:
- MongoClient(...) dipakai sebagai nilai default parameter di MongoRepository.__init__, sehingga koneksi dibuat saat kelas didefinisikan, bukan saat instance dibuat. Lebih baik memakai None lalu membuat klien di dalam __init__.
- Sel 31 dan 32 memodifikasi database langsung, jadi dijalankan ulang akan mengubah data lagi.
- Isi dokumen berupa data pribadi (nama, email, tanggal lahir). Pastikan ekspor CSV hanya memuat kolom yang diperlukan, seperti yang sudah dilakukan (hanya email dan tag).
- Nama file CSV memakai tanggal hari ini (Timestamp.now()), bukan tanggal data yang diproses, sehingga bisa membingungkan jika dijalankan pada hari yang berbeda.